In [1]:
import os
import pandas as pd
import numpy as np
import re
import string
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from scipy.sparse import hstack

import lightgbm as lgb
import optuna

In [2]:
df = pd.read_csv("../data/raw/recipes.csv")

df["text"] = (
    df["Name"].fillna('') + " " +
    df["RecipeIngredientParts"].fillna('') + " " +
    df["RecipeInstructions"].fillna('')
)

df = df.dropna(subset=["AggregatedRating"]).reset_index(drop=True)

In [3]:
def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    return text

df["clean_text"] = df["text"].apply(preprocess)

In [4]:
tfidf = TfidfVectorizer(ngram_range=(1,1), max_features=5000)
X_tfidf = tfidf.fit_transform(df["clean_text"])

lsa = TruncatedSVD(n_components=100, random_state=0)
X_lsa = lsa.fit_transform(X_tfidf)

count_vec = CountVectorizer(max_features=5000)
X_count = count_vec.fit_transform(df["clean_text"])
lda = LatentDirichletAllocation(n_components=20, random_state=0)
X_lda = lda.fit_transform(X_count)

X_final = hstack([X_tfidf, X_lsa, X_lda]).tocsr()

In [5]:
y = df["AggregatedRating"].values

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42
)

In [12]:
def objective(trial):
    param = {
        "objective": "regression",
        "metric": "rmse",
        "verbosity": -1,
        "boosting_type": "gbdt",
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.1, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 2, 256),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
    }
    
    gbm = lgb.LGBMRegressor(**param)
    gbm.fit(X_train, y_train, eval_set=[(X_test, y_test)])
    
    preds = gbm.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, preds))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30, n_jobs=-1)

print(f"Best RMSE: {study.best_trial.value}")

[I 2026-03-27 22:44:55,423] A new study created in memory with name: no-name-3bb8eb54-aae3-410b-ad83-7abcb8ffc984
/opt/homebrew/anaconda3/envs/SE481/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-27 22:46:40,066] Trial 1 finished with value: 0.6386837674761829 and parameters: {'n_estimators': 118, 'learning_rate': 0.0026812918470580666, 'num_leaves': 13, 'subsample': 0.46754956710348733, 'colsample_bytree': 0.792885331721446, 'min_child_samples': 19}. Best is trial 1 with value: 0.6386837674761829.
/opt/homebrew/anaconda3/envs/SE481/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
[I 2026-03-27 22:56:22,998] Trial 4 finished with value: 0.6324003387378706 and parameters: {'n_estimators': 184, 'learning_rate': 0.05064531

Best RMSE: 0.6307625584919334


In [7]:
best_model = lgb.LGBMRegressor(**study.best_trial.params)
best_model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.236231 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 710622
[LightGBM] [Info] Number of data points in the train set: 215435, number of used features: 5111
[LightGBM] [Info] Start training from score 4.631040


,boosting_type,'gbdt'
,num_leaves,102
,max_depth,-1
,learning_rate,0.02944304042220195
,n_estimators,459
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,41


In [8]:
with open('../resources/recipe_recommendation_model.pkl', 'wb') as f:
    pickle.dump({
        'model': best_model,
        'tfidf': tfidf,
        'lsa': lsa,
        'count_vec': count_vec,
        'lda': lda,
        'X_final': X_final 
    }, f)

In [9]:
def recommend_for_user(user_recipe_ids, top_k=10):
    indices = df[df["RecipeId"].isin(user_recipe_ids)].index
    
    if len(indices) == 0:
        return "No valid user recipes"
        
    user_profile = X_final[indices].mean(axis=0)
    similarity_scores = np.asarray(X_final @ user_profile.T).flatten()
    
    predicted_ratings = best_model.predict(X_final)
    
    scaler = MinMaxScaler()
    normalized_similarity = scaler.fit_transform(similarity_scores.reshape(-1, 1)).flatten()
    normalized_ratings = scaler.fit_transform(predicted_ratings.reshape(-1, 1)).flatten()
    
    df["final_score"] = (normalized_similarity * 0.7) + (normalized_ratings * 0.3)
    
    recs = df.drop(indices).sort_values("final_score", ascending=False)
    return recs[["RecipeId", "Name", "AggregatedRating", "final_score"]].head(top_k)

In [13]:
sample_recipes = df.sample(5)
sample_ids = sample_recipes["RecipeId"].tolist()

display(sample_recipes[["RecipeId", "Name"]])
display(recommend_for_user(sample_ids))

,RecipeId,Name
8553,14435,Squirnip
80055,112118,Yellow Maize Flour and Fenugreek Leaves Indian...
266069,507136,Brazilian-Style Beans
34908,49284,Swiss Pears
132606,197565,Emily's Easy Peanut Butter Candy!


/opt/homebrew/anaconda3/envs/SE481/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,RecipeId,Name,AggregatedRating,final_score
110636,160537,Slow-Cooked White Beans,5.0,0.837314
114236,166703,Mixed Bean Chili,5.0,0.831140
132846,198003,Sauteed Green Beans and Red Onion,5.0,0.822552
59970,83612,Jamaica Me Crazy Chili,5.0,0.821438
243486,427708,Quick Southwestern Black Beans,5.0,0.820149
131687,195907,Pumpkin Chili,4.0,0.809919
238724,415168,Crispy Bacon Green Beans,5.0,0.806458
2776,6555,Red Beans With Rice,5.0,0.806315
142793,215362,Spicy Vegetarian Chili,5.0,0.806124
209540,345162,Paul's Vegetarian Chili,5.0,0.805884
